In [ ]:
!pip install datasets transformers torch tqdm

import numpy as np
import torch
from datasets import load_dataset
from transformers import DistilBertTokenizer, DistilBertModel
from tqdm import tqdm

device = torch.device("cuda")

# Load dataset
dataset = load_dataset("ag_news")

train = dataset["train"].shuffle(seed=42).select(range(8000))
test = dataset["test"].shuffle(seed=42).select(range(2000))

train_texts = train["text"]
test_texts = test["text"]

# Load model
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = DistilBertModel.from_pretrained("distilbert-base-uncased").to(device)
model.eval()

def get_embeddings(texts):
    embeddings = []
    for i in tqdm(range(0, len(texts), 32)):
        batch = texts[i:i+32]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            cls = outputs.last_hidden_state[:, 0, :]

        embeddings.append(cls.cpu().numpy())

    return np.vstack(embeddings)

# Extract
train_emb = get_embeddings(train_texts)
test_emb = get_embeddings(test_texts)

# Save
np.save("train.npy", train_emb)
np.save("test.npy", test_emb)

print("Saved embeddings!")